# Import

In [10]:
import pandas as pd

# df1 = pd.read_csv("../../data/dataset.csv", sep="\\|\\|\\|", engine="python").drop(["modern_prompt", "translation_prompt"], axis=1)
df1 = pd.read_csv("../../data/cleaned_dataset.csv", sep="|", engine="python")
df2_init = pd.read_csv("../../data/paires_mot_lemme.csv", )
df2 = df2_init.rename(columns={'mot': 'old_french', 'lemme': 'modern'})

In [11]:
# df1["modern"] = df1["modern"]
# df1["old_french"] = df1["old_french"]
df1.head()

,modern,old_french
0,"salut tout le monde, c'est victor. je repensai...","seigneurs et dames, je vous salue, c'est victo..."
1,"salut tout le monde ! aujourd'hui, c'était un ...","saluz tout le monde ! en ce jour, fu moult bon..."
2,"chère sophie, tu ne devineras jamais ce qui m'...","chère sophie, tu ne devineras ja mie ce qui m'..."
3,"bonjour à tous, ici marcel dupré, artisan poti...","bien le bon jour à tous, céans marcel dupré, o..."
4,"yo, c’est lila. alors voilà, l’autre jour, j’é...","salut, c'est lila. lors, l'autre jour, j'estoi..."


In [12]:
df2.head(10)

,old_french,modern
0,Cil,cil
1,qui,qui1
2,fist,faire
3,d',de
4,Erec,Erec
5,et,et
6,Enide,Enide
7,Et,et
8,les,le
9,comandemanz,comandement


In [13]:
df = pd.concat([df1, df2], ignore_index=True)

In [14]:
import pandas as pd

# Charger le fichier CSV avec le bon séparateur
# df = pd.read_csv("../../data/dataset.csv", sep="\\|\\|\\|", engine="python")

# Ne conserver que les colonnes utiles
df = df[['modern', 'old_french']].dropna()

# Nettoyage basique (optionnel mais recommandé)
df['modern'] = df['modern'].str.strip()
df['old_french'] = df['old_french'].str.strip()

# Vérification
print(df.sample(3))

        modern old_french
27110  affier1       afié
1400     maint      Maint
8048     Autun   Hostedun


In [15]:
df.shape

(30111, 2)

# Vieux truc avec cartes graphique pas opti

In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("t5-small", use_fast=False)

/home/malek/BRIEFS DEV IA/14.NLP/EULA-vaaag-/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [8]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch

class TranslationDataset(Dataset):
    def __init__(self, modern_texts, old_french_texts, tokenizer, max_length=128):
        self.modern_texts = modern_texts
        self.old_french_texts = old_french_texts
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.modern_texts)
    
    def __getitem__(self, idx):
        modern = self.modern_texts[idx]
        old_french = self.old_french_texts[idx]
        modern_text = f"translate French to OldFrench: {modern}"
        
        # Tokenisation
        inputs = self.tokenizer(
            modern_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        targets = self.tokenizer(
            old_french, 
            max_length=self.max_length, 
            padding='max_length', 
            truncation=True, 
            return_tensors='pt'
        )
        
        return {
            # 'input_ids': inputs['input_ids'].flatten(),
            'input_ids': inputs['input_ids'].squeeze(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'decoder_attention_mask': targets['attention_mask'].squeeze(),
            'labels': targets['input_ids'].flatten()
        }

# Modèle basé sur mT5 ou mBERT
model = AutoModelForSeq2SeqLM.from_pretrained('t5-small')

2025-06-19 13:32:20.677354: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-19 13:32:20.684803: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750332740.693542  113619 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750332740.696278  113619 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750332740.703303  113619 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [9]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
test_df, val_df = train_test_split(val_df, test_size=0.5, random_state=42)

In [10]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=6,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    # gradient_accumulation_steps=4,
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=True,
    dataloader_pin_memory=False,
)


In [11]:
train_dataset = TranslationDataset(
    modern_texts=train_df["modern"].tolist(),
    old_french_texts=train_df["old_french"].tolist(),
    tokenizer=tokenizer,
    max_length=160
)

val_dataset = TranslationDataset(
    modern_texts=val_df["modern"].tolist(),
    old_french_texts=val_df["old_french"].tolist(),
    tokenizer=tokenizer,
    max_length=160
)


In [12]:
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer
import nltk
import numpy as np
from typing import Dict
import torch

def evaluate_model(predictions, references):
    # BLEU Score
    bleu = corpus_bleu(predictions, [references])
    
    # ROUGE Score
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = [scorer.score(ref, pred) for ref, pred in zip(references, predictions)]
    
    # Similarité de caractères (important pour l'ancien français)
    char_similarity = [
        len(set(ref) & set(pred)) / len(set(ref) | set(pred)) if len(set(ref) | set(pred)) > 0 else 0
        for ref, pred in zip(references, predictions)
    ]
    
    return {
        'bleu': bleu.score,
        'rouge1': np.mean([s['rouge1'].fmeasure for s in rouge_scores]),
        'char_similarity': np.mean(char_similarity)
    }

def compute_metrics(eval_pred) -> Dict:
    predictions, labels = eval_pred
    
    # CORRECTION: Convertir les logits en tokens IDs
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    
    # Si predictions contient des logits, prendre l'argmax
    if len(predictions.shape) == 3:  # [batch, seq_len, vocab_size]
        predictions = np.argmax(predictions, axis=-1)
    
    # Remplacer les labels -100 par le token pad
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    # Décoder les prédictions et labels
    try:
        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        
        # Nettoyer les textes vides
        decoded_preds = [pred.strip() if pred.strip() else "vide" for pred in decoded_preds]
        decoded_labels = [label.strip() if label.strip() else "vide" for label in decoded_labels]
        
        return evaluate_model(decoded_preds, decoded_labels)
    
    except Exception as e:
        print(f"Erreur dans compute_metrics: {e}")
        print(f"Shape predictions: {predictions.shape}")
        print(f"Shape labels: {labels.shape}")
        print(f"Type predictions: {type(predictions)}")
        
        # Retourner des métriques par défaut en cas d'erreur
        return {
            'bleu': 0.0,
            'rouge1': 0.0,
            'char_similarity': 0.0
        }


In [13]:
print(torch.cuda.get_device_name(0)) 

NVIDIA GeForce RTX 4060 Laptop GPU


In [14]:
import torch

torch.cuda.empty_cache()           # Libère la mémoire inutilisée (mais allouée)
torch.cuda.ipc_collect()           # Nettoie les handles CUDA obsolètes (utile en Notebook)

# Affiche l'utilisation mémoire
print(f"Mémoire GPU utilisée: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Mémoire GPU réservée: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

Mémoire GPU utilisée: 0.00 GB
Mémoire GPU réservée: 0.00 GB


In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipykernel_113619/4030190838.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.14 GiB. GPU 0 has a total capacity of 7.75 GiB of which 3.07 GiB is free. Including non-PyTorch memory, this process has 4.62 GiB memory in use. Of the allocated memory 3.88 GiB is allocated by PyTorch, and 631.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# enregistrer le modele

model_save_path = "./model_save2"

trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

('./model_save/tokenizer_config.json',
 './model_save/special_tokens_map.json',
 './model_save/spiece.model',
 './model_save/added_tokens.json')

In [ ]:
# Exemple d'entrée
modern_text = "Salut tout le monde ! Aujourd'hui on a fait un pique-nique dans le jardin. Après j'ai été malade, je me suis vidé par tous les trous.... ça me fait chier d'être malade comme ça j'en peux plus je suis au bout de ma vie !"
input_text = f"translate French to OldFrench: {modern_text}"

# Tokenisation
inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)

# Déplacer explicitement les tenseurs d'entrée vers le GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = {name: tensor.to(device) for name, tensor in inputs.items()}

# Génération (inférence)
with torch.no_grad():
    output_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

# Décodage
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("📝 Texte d'origine :", modern_text)
print("🏰 Traduction en vieux français :", output_text)


📝 Texte d'origine : Salut tout le monde ! Aujourd'hui on a fait un pique-nique dans le jardin. Après j'ai été malade, je me suis vidé par tous les trous.... ça me fait chier d'être malade comme ça j'en peux plus je suis au bout de ma vie !
🏰 Traduction en vieux français : Saluz à tous le monde! En ce jour, nous avons fait ung pique-nique en le jardin. Après j'ay été malade, je me suis vidé par tous les trous.... ce me fait chier d'estre malade comme ça j'en peux plus je suis au bout de ma vie!
